# Marketplace exploratory analysis

Working notebook used to explore the data before anything was promoted into `src/`.
Outputs are cleared before committing so diffs stay readable.

Questions I set out to answer:

1. Is revenue growing, and is that growth coming from more customers or bigger baskets?
2. How concentrated is revenue across customers?
3. Which categories carry volume but not margin?
4. Does an early second purchase predict long-term value?

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import matplotlib.pyplot as plt

from src.db import load_orders
from src.analysis import kpis, rfm, cohorts

pd.set_option('display.float_format', lambda v: f'{v:,.2f}')

orders = load_orders()
orders.shape

## 1. Sanity checks first

Before any analysis: are there duplicates, negative values or gaps in the date range?
Every number downstream depends on these being clean.

In [ ]:
checks = {
    'rows': len(orders),
    'distinct_orders': orders['order_id'].nunique(),
    'distinct_customers': orders['customer_id'].nunique(),
    'date_range': (orders['order_date'].min(), orders['order_date'].max()),
    'null_customer_ids': int(orders['customer_id'].isna().sum()),
    'non_positive_revenue': int((orders['net_revenue'] <= 0).sum()),
}
checks

## 2. Headline KPIs and monthly trend

In [ ]:
kpis.headline_summary(orders)

In [ ]:
monthly = kpis.monthly_kpis(orders)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(monthly['order_month'], monthly['net_revenue'], marker='o')
axes[0].set_title('Net revenue by month')
axes[1].plot(monthly['order_month'], monthly['aov'], marker='o', color='darkorange')
axes[1].set_title('Average order value')
for ax in axes:
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()

Growth is coming from **customer count**, not basket size — AOV is flat. That points the
conversation at retention rather than at pricing.

## 3. How concentrated is revenue?

In [ ]:
segmented = rfm.segment_customers(orders)

for pct in (0.05, 0.10, 0.20, 0.50):
    print(rfm.revenue_concentration(segmented, top_pct=pct))

In [ ]:
summary = rfm.segment_summary(segmented)
summary

The **At Risk** and **Cannot Lose Them** segments together hold a large share of historical
revenue and have not purchased recently. That is the single highest-ROI place to spend
marketing budget, and it was completely invisible in the old monthly report.

## 4. Category margin vs volume

In [ ]:
category = kpis.category_performance(orders)

ax = category.plot.scatter(x='units', y='margin_pct', s=80, figsize=(7, 5))
for _, row in category.iterrows():
    ax.annotate(row['category'], (row['units'], row['margin_pct']), xytext=(6, 4), textcoords='offset points')
ax.set_title('High volume, low margin categories sit bottom-right')
category

## 5. Does a fast second purchase predict value?

In [ ]:
cohorts.second_purchase_window(orders, window_days=45)

In [ ]:
retention = cohorts.retention_matrix(orders)
retention.round(1).head(12)

## Takeaways promoted into the dashboard

- Revenue concentration and the segment table -> **page 2**
- Cohort retention heatmap and the 45-day finding -> **page 3**
- Category margin scatter -> **page 4**

Anything that only mattered once stayed in this notebook. Only the recurring questions
became dashboard pages — otherwise the report becomes a dumping ground nobody reads.